# 1단계 · FAISS로 도서관 대출 규정에 답하기

가상의 단풍도서관에서 **일반 도서의 기본 대출 기간은 며칠인가요?** 서로 다른 안내문 네 개를 문단 단위로 나누고, FAISS가 찾은 상위 청크 세 개를 한 번에 생성 모델에 전달한다. 한국어 질문, 검색 결과, 프롬프트, 답변을 함께 기록해 이후 실험의 기준으로 쓴다.

Python 3 환경에서 필요한 경우 아래 설치 명령의 주석을 풀고, 위에서부터 모든 셀을 실행한다. 처음 실행할 때 공개 모델을 내려받는다. 캐시에 모델이 있으면 CPU에서도 오프라인으로 실행할 수 있다.

In [1]:
# %pip install faiss-cpu scikit-learn numpy torch "transformers>=4.51"

## 문서와 청크

각 문서는 다른 상황에서 독자가 접하는 안내문이다. 빈 줄로 나뉜 문단 하나를 청크 하나로 사용한다. 청크 ID 앞부분은 원문 문서 ID이고 뒷부분은 그 안의 문단 번호다.

In [2]:
documents = {
    "이용규정": """일반 도서의 기본 대출 기간은 14일입니다.

참고 도서는 도서관 안에서만 열람할 수 있습니다.""",
    "회원안내": """회원은 일반 도서를 14일 동안 빌릴 수 있습니다.

도서관 회원증을 신청하려면 안내 데스크에 사진이 있는 신분증을 제시하세요.""",
    "대출영수증": """일반 도서의 대출 기한은 대출일로부터 14일입니다.

대출 영수증에는 빌린 책의 제목과 반납 예정일이 표시됩니다.""",
    "반납안내": """야외 도서 반납함은 도서관 정문 바로 옆에 있습니다.

대형 자료는 안내 데스크에 직접 반납해 주세요.""",
}

chunks = [
    {"id": f"{doc_id}/{number}", "text": paragraph.strip()}
    for doc_id, text in documents.items()
    for number, paragraph in enumerate(text.split("\n\n"), start=1)
    if paragraph.strip()
]
print(f"문서 {len(documents)}개 → 청크 {len(chunks)}개")
for chunk in chunks:
    print(f"{chunk['id']} · {chunk['text']}")

문서 4개 → 청크 8개
이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
이용규정/2 · 참고 도서는 도서관 안에서만 열람할 수 있습니다.
회원안내/1 · 회원은 일반 도서를 14일 동안 빌릴 수 있습니다.
회원안내/2 · 도서관 회원증을 신청하려면 안내 데스크에 사진이 있는 신분증을 제시하세요.
대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
대출영수증/2 · 대출 영수증에는 빌린 책의 제목과 반납 예정일이 표시됩니다.
반납안내/1 · 야외 도서 반납함은 도서관 정문 바로 옆에 있습니다.
반납안내/2 · 대형 자료는 안내 데스크에 직접 반납해 주세요.


## FAISS 검색

조사가 붙어도 ‘반납함’과 ‘반납함은’의 공통 부분을 찾을 수 있도록 어절 안에서 3~5글자씩 겹쳐 읽는다. 어절 양끝의 공백도 글자 수에 포함한다. 질문과 청크를 이 문자 조각의 TF-IDF 벡터로 표현하고 L2 정규화한다. FAISS가 계산한 내적은 코사인 유사도이며, 점수가 0보다 큰 청크를 최대 $k$개 반환한다.

In [3]:
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

class PassageIndex:
    def __init__(self, chunks):
        self.chunks = list(chunks)
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), dtype=np.float32, norm="l2"
        )
        matrix = self.vectorizer.fit_transform([chunk["text"] for chunk in self.chunks])
        self.index = faiss.IndexFlatIP(matrix.shape[1])
        self.index.add(matrix.toarray())

    def search(self, question, k=3):
        if k <= 0:
            return []
        query = self.vectorizer.transform([question])
        if query.nnz == 0:
            return []
        scores, positions = self.index.search(query.toarray(), min(k, self.index.ntotal))
        return [
            {**self.chunks[position], "score": float(score)}
            for score, position in zip(scores[0], positions[0])
            if score > 0
        ]

retriever = PassageIndex(chunks)
question = "일반 도서의 기본 대출 기간은 며칠인가요?"
hits = retriever.search(question, k=3)

print(question)
for rank, hit in enumerate(hits, start=1):
    print(f"{rank}. [{hit['score']:.3f}] {hit['id']} · {hit['text']}")

일반 도서의 기본 대출 기간은 며칠인가요?
1. [0.772] 이용규정/1 · 일반 도서의 기본 대출 기간은 14일입니다.
2. [0.297] 대출영수증/1 · 일반 도서의 대출 기한은 대출일로부터 14일입니다.
3. [0.084] 회원안내/1 · 회원은 일반 도서를 14일 동안 빌릴 수 있습니다.


## 검색한 근거로 답변 생성

상위 청크들을 하나의 한국어 프롬프트에 담아 Qwen3-0.6B를 한 번 호출한다. 같은 조건에서 결과를 비교할 수 있도록 모델 버전을 고정하고, 비사고 모드에서 매번 확률이 가장 높은 토큰을 선택한다. 검색 결과가 비면 모델 호출을 건너뛴다. [Qwen3-0.6B 모델 카드](https://huggingface.co/Qwen/Qwen3-0.6B)

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as model_logging

model_logging.disable_progress_bar()
torch.set_num_threads(min(torch.get_num_threads(), 4))
model_id = "Qwen/Qwen3-0.6B"
revision = "c1899de289a04d12100db370d81485cdf75e47ca"
tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)
model = AutoModelForCausalLM.from_pretrained(
    model_id, revision=revision, use_safetensors=True
).eval()

def make_prompt(question, passages):
    context = "\n".join(passage["text"] for passage in passages)
    return (
        f"다음은 도서관 안내문입니다.\n\n{context}\n\n질문: {question}\n"
        "안내문에서 질문의 답을 찾아 한국어로 한 문장으로 답하세요."
    )

@torch.inference_mode()
def answer_from(question, passages):
    if not passages:
        return "근거 없음"
    messages = [{"role": "user", "content": make_prompt(question, passages)}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        return_tensors="pt", return_dict=True,
    )
    output = model.generate(**inputs, do_sample=False, max_new_tokens=48)
    answer_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

print(make_prompt(question, hits))
answer = answer_from(question, hits)
print(f"\n모델 답변 → {answer}")

다음은 도서관 안내문입니다.

일반 도서의 기본 대출 기간은 14일입니다.
일반 도서의 대출 기한은 대출일로부터 14일입니다.
회원은 일반 도서를 14일 동안 빌릴 수 있습니다.

질문: 일반 도서의 기본 대출 기간은 며칠인가요?
안내문에서 질문의 답을 찾아 한국어로 한 문장으로 답하세요.



모델 답변 → 일반 도서의 기본 대출 기간은 14일입니다.


## 기준 결과 확인

대출 기간을 뒷받침하는 청크 세 개가 모두 검색됐는지, 생성 답변에 올바른 기간과 반납함 위치가 담겼는지 검사한다. 문서에 없는 망원경 질문은 검색 결과가 비어 ‘근거 없음’으로 처리되는지도 확인한다.

In [5]:
assert retriever.index.ntotal == len(chunks) == 8
assert hits[0]["id"] == "이용규정/1"
assert {hit["id"] for hit in hits} == {"이용규정/1", "회원안내/1", "대출영수증/1"}
assert "14일" in answer.replace(" ", "") or "열나흘" in answer, answer

location_question = "야외 도서 반납함은 어디에 있나요?"
location_hits = retriever.search(location_question)
location_answer = answer_from(location_question, location_hits)
print(f"{location_question} → {location_answer} [{location_hits[0]['id']}]")
assert location_hits[0]["id"] == "반납안내/1"
assert "정문" in location_answer and "옆" in location_answer, location_answer

unknown_question = "망원경이 있나요?"
unknown_hits = retriever.search(unknown_question)
unknown_answer = answer_from(unknown_question, unknown_hits)
print(f"{unknown_question} → {unknown_answer} (검색 {len(unknown_hits)}건)")
assert not unknown_hits and unknown_answer == "근거 없음"

야외 도서 반납함은 어디에 있나요? → 야외 도서 반납함은 도서관 정문 바로 옆에 있습니다. [반납안내/1]
망원경이 있나요? → 근거 없음 (검색 0건)
